# Risk definition for residual load

Implements [`.claude/specs/02-Risk-Definition.md`](../../.claude/specs/02-Risk-Definition.md).

`notebooks/01_eda/team-EDA.ipynb` closed with one explicit open question: the two extremes of
`residual_load` are not one phenomenon — they differ in season, hour, day type, trend and
mechanism — and *"we need to decide whether the risk flag should treat them as one target or
two"*. This notebook answers that question and turns the answer into a concrete, reproducible
labelling methodology.

**This is not an EDA notebook.** Every finding it leans on was already established in
`team-EDA.ipynb` and is restated here in one or two lines, never re-derived.

## What this notebook produces

- Three **threshold bases** for each direction, derived from `data/smard.csv` itself with no
  hardcoded MWh figure: a causal `rolling` trailing-window quantile, a physical `zero` crossing
  (low direction only), and a whole-record `static` quantile kept as a reference.
- **Definition 1** — a per-day risk flag with a reason (the triggering hour and its value), under
  two day rules: `any` and a 3-hour persistence rule.
- **Definition 2** — per-hour flags using those same thresholds, and a derived time range per
  flagged day-direction.
- Two exported artifacts, `data/risk_labels_daily.csv` and `data/risk_labels_hourly.csv`, which
  the modeling spec can load directly.

## Conventions

- `time_series` is the main dataframe, `SERIES` its measured columns, `DERIVED` the engineered
  ones — inherited unchanged from [`03.1-setup.md`](../../.claude/specs/03.1-setup.md).
- `YEARS` is computed at run time; no literal calendar year appears in code.
- **Units:** `MWh` for every level and threshold. The dataset is energy data for the whole grid,
  so an hourly reading is treated as energy, not power.
- **Durations, never row counts.** Every duration-based rule below is expressed as a duration
  ("at least 3 hours"), not as a number of observations ("at least 3 rows"), so a later switch to
  SMARD's 15-minute resolution would not silently change the rule's meaning.

---

## 1 Setup

Inherited verbatim from [`03.1-setup.md`](../../.claude/specs/03.1-setup.md), the spec behind
`team-EDA.ipynb`'s setup section: the data-directory resolver, `time_series`, `SERIES`, `DERIVED`,
`YEARS`, `period_mean` / `period_energy`, `style_timeseries`, `DAY_NAMES` and the season mapping.
Nothing here is re-derived — see that spec and `notebooks/01_eda/team-EDA.ipynb` §1 for the
reasoning behind each piece.

`data/` is **gitignored**, so `data/smard.csv` does not come with a clone. Regenerate it by
running [`notebooks/API-connection.ipynb`](../API-connection.ipynb) top to bottom.

The data directory is resolved by walking **upward** from the working directory rather than by a
fixed `"../../data/smard.csv"`, so the notebook runs unmodified whether the kernel starts in
`notebooks/03_risk_classification/` or at the repo root.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Walk up from the working directory to the first parent holding a `data/` folder
DATA_DIR = next(
    (p / "data" for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(
        f"no data/ directory found in {Path.cwd()} or any parent — start the kernel inside the "
        "repository, then re-run."
    )
DATA = DATA_DIR / "smard.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running notebooks/API-connection.ipynb top to bottom."
    )

print(f"pandas {pd.__version__} · numpy {np.__version__}")
print(f"Data directory: {DATA}")

### 1.1 Helpers

Copied unchanged from `team-EDA.ipynb` §1.1. `_complete_periods` drops calendar periods the data
does not fully cover — a plain `.resample()` produces fake edge dips. `style_timeseries` requires
an explicit `ylabel`, so no plot can ship without stating its unit.

In [ ]:
def _complete_periods(index, freq):
    """The calendar periods of `freq` that `index` covers completely.

    A period counts only if it starts not earlier than the first observation and ends not later than the last observation's closing edge.
    Both `period_mean` and `period_energy` defer to this.
    """
    periods = index.to_period(freq).unique().sort_values()
    complete = (
        periods.start_time >= index.min()) & (
        periods.end_time <= index.max() + pd.Timedelta("1h")
    )
    return periods[complete]


def period_mean(series, freq):
    """Mean of `series` per calendar period (`"W"`, `"M"`, ...), indexed by period start.

    Periods that the data does not cover completely are dropped, so the edges of a plot are not
    partial-period artefacts (half weeks / months, starting or ending a week on thursday).
    A period is considered an artifact if the range is "not fully covered".
    """
    agg = series.groupby(series.index.to_period(freq)).mean()
    agg = agg.loc[_complete_periods(series.index, freq)]
    agg.index = agg.index.start_time
    return agg


def period_energy(series, freq, drop_incomplete=True):
    """Per-period aggregate of `series` in both project reporting units.

    Returns a DataFrame indexed by period start:

    ``mwh_per_day``
        period sum / calendar days in the period — the energy view (MWh/day).
    ``avg_mw``
        period sum / hours **actually present** — the level view (MW).
        Deliberately not ``mwh_per_day / 24``: a month containing the spring Daylight-Saving-Time switch holds 743 hours, not 744.
    ``hours``, ``days``
        the two denominators, exposed so a comparison table needs no second copy of this
        arithmetic.

    Incomplete periods are dropped by the same `_complete_periods` rule as `period_mean`.
    """
    grouped = series.groupby(series.index.to_period(freq))
    total, hours = grouped.sum(), grouped.size()
    periods = total.index

    if drop_incomplete:
        keep = _complete_periods(series.index, freq)
        total, hours, periods = total.loc[keep], hours.loc[keep], keep

    # Freq-generic: 7 for every week, 28-31 for months. `days_in_month` would be "M"-only.
    days = (periods.end_time.normalize() - periods.start_time).days + 1

    # .to_numpy() on every right-hand side: aligning a PeriodIndex-backed Series against a
    # DatetimeIndex-derived array silently yields all-NaN.
    return pd.DataFrame(
        {
            "mwh_per_day": total.to_numpy() / days,
            "avg_mw": total.to_numpy() / hours.to_numpy(),
            "hours": hours.to_numpy(),
            "days": np.asarray(days),
        },
        index=periods.start_time,
    )

In [ ]:
def style_timeseries(ax, title, ylabel):
    """Custom grid, no box, year ticks.

    `ylabel` is required: every plot must state whether it shows MWh, average MW or MWh/day.
    """
    ax.set_title(
        title,
        loc="center",
        fontsize=15,
        pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        ylabel,
        color="grey"
    )
    ax.grid(
        axis="y",
        color="0.9",
        linewidth=0.8
    )
    ax.set_axisbelow(True)

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    ax.tick_params(
        colors="black",
        length=0  # hide ticks of values
    )
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")

### 1.2 Colour configuration

The subset of `team-EDA.ipynb` §1.2 this notebook needs. `TAIL_COLOR` carries over unchanged, so
the two directions read the same here as they do in the EDA: cool blue for the low/oversupply
tail, hot red for the high/undersupply tail.

In [ ]:
COLORS = {
    "blue":  "#2C6EBA",
    "red":   "#B10F0F",
    "black": "#1C1C1C",
    "gray":  "#AEB8C5",
    "muted": "#707B8C",
}

# Residual-load extremes, wherever they are shown as a pair:
# cool for oversupply (low), hot for undersupply (high).
TAIL_COLOR = {
    "low": COLORS["blue"],
    "high": COLORS["red"],
}

# One line style per threshold basis, used on every threshold plot below.
BASIS_STYLE = {
    "rolling": {"linestyle": "-", "linewidth": 2.0},
    "static": {"linestyle": "--", "linewidth": 1.4},
    "zero": {"linestyle": ":", "linewidth": 1.4},
}

DIRECTION_LABEL = {"high": "High residual load", "low": "Low / negative residual load"}

print(f"{len(TAIL_COLOR)} directions, {len(BASIS_STYLE)} bases configured")

### 1.3 Load and prepare

Dtype assertion guards against the German Excel-CSV conversion silently leaving a column as text.
Everything after this cell uses `time_series`.

In [ ]:
# The CSV headers exactly as notebooks/API-connection.ipynb writes them.
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid Load": "fc_grid_load",
    "Forecast Residual Load": "fc_residual_load",
}

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")

assert set(raw.columns) == {"timestamp"} | set(COLUMNS), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | set(COLUMNS)))}"
)

raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

time_series = raw.set_index("timestamp").sort_index()
del raw  # the flat frame does not outlive the loading cell

# Positive is_float_dtype test, not `!= object`: under pandas 3 an unconverted German-decimal
# column lands as StringDtype, and `!= object` would wave it straight through.
assert all(
    pd.api.types.is_float_dtype(time_series[c]) for c in COLUMNS.values()
), time_series.dtypes

# Snapshot taken before any other cell can touch the frame, so the closing self-check can prove
# nothing in between mutated it.
LOADED = {
    "rows": len(time_series),
    "start": time_series.index.min(),
    "end": time_series.index.max(),
}

print(f"shape           : {time_series.shape[0]:,} rows x {time_series.shape[1]} columns")
print(f"index           : {time_series.index.min()}  ->  {time_series.index.max()}")
print(
    f"index monotonic : {time_series.index.is_monotonic_increasing}, "
    f"unique: {time_series.index.is_unique}"
)
time_series.head(3)

### 1.4 Derived columns, `SERIES` / `DERIVED`, and `YEARS`

`YEARS` is computed from the loaded data and is the only permitted source of year information in
the rest of the notebook. `spans_gap` marks the row *following* a gap in the hourly index — the
record's only gaps are the spring Daylight-Saving-Time switches, where the local hour 02:00 does
not exist.

In [ ]:
SERIES = [
    "wind_off", "wind_on", "solar", "grid_load", "residual_load",
    "fc_gen_wind_solar", "fc_grid_load", "fc_residual_load",
]

DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

time_series["renewables"] = time_series[["wind_on", "wind_off", "solar"]].sum(axis=1)
time_series["year"] = time_series.index.year
time_series["month"] = time_series.index.month
time_series["hour"] = time_series.index.hour
time_series["dow"] = time_series.index.dayofweek
time_series["is_weekend"] = time_series.index.dayofweek >= 5
time_series["date"] = time_series.index.date
time_series["season"] = pd.Categorical(
    time_series.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
time_series["season_year"] = time_series.index.year + (time_series.index.month == 12)

# Outputs True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
time_series["spans_gap"] = time_series.index.to_series().diff() > pd.Timedelta("1h")

DERIVED = [
    "renewables", "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

# Plain ints, not np.int32: they end up in titles, labels and dict keys all over the notebook.
YEARS = sorted(int(y) for y in time_series["year"].unique())

print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {time_series.shape[1]} columns")
print(f"YEARS = {YEARS}")

### 1.5 Initial self-check

Structural only, and deliberately free of any hardcoded row count or date bound: the record's
extent is expected to change. The closing self-check re-runs these invariants plus a comparison
against `LOADED`.

In [ ]:
assert all(pd.api.types.is_float_dtype(time_series[c]) for c in SERIES), time_series[SERIES].dtypes
assert time_series.index.is_monotonic_increasing, "index is not sorted"
assert time_series.index.is_unique, "index has duplicate timestamps"
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert YEARS == sorted(int(y) for y in time_series["year"].unique())

print("setup self-check passed")
print(f"  {len(SERIES)} float series, index sorted and unique")
print(f"  columns == SERIES + DERIVED ({len(SERIES) + len(DERIVED)} columns)")
print(f"  {LOADED['rows']:,} rows, {LOADED['start']} -> {LOADED['end']}, years {YEARS}")

### 1.6 What this notebook inherits from `team-EDA.ipynb`

Three findings carry real weight here. They are **cited, not re-derived** — no new evidence is
gathered for them in this notebook.

- **§3.7 — the two tails have almost mirror-image calendar signatures.** The low/negative extreme
  is overwhelmingly recent, concentrated in high-solar months, midday hours and weekends (the
  weekly demand minimum lining up with the daily solar maximum); the high extreme is weekday-only,
  spread evenly across years, in winter months, peaking in the evening with a secondary morning
  peak. §3.7 reads the low extreme as *growing and seasonal* and the high extreme as *stable and
  structural*. **This is the direct justification for treating high and low as two independently
  thresholded directions rather than one signed scale.** §3.7 also selected its 1 % tail *by rank*
  and labelled it explicitly a descriptive slice, not a risk definition.
- **§6.3 — the distribution is close to symmetric with a mild negative skew, and 1.26 % of all
  hours are already negative** — a small share, but growing and highly clustered in summer. There
  is no second mode and no sharp cutoff, so every threshold question here is a judgement call
  rather than a boundary the data hands us.
- **§6.5 — the low tail is stretching downward while the high end barely moves.** P10 falls faster
  than the median year on year; P90 hardly moves at all. This notebook's basis comparison exists
  specifically to make that asymmetry visible in threshold terms.

§6.5's **matched calendar window** (1 January to the record's last complete date, applied
identically to every year, derived from the data and never hardcoded) is the only admissible way
to compare years in this notebook — see §2 on why.

### 1.7 The residual-load identity

Before thresholding `residual_load`, confirm what it actually is in this record:

$$\text{residual\_load} = \text{grid\_load} - (\text{wind\_off} + \text{wind\_on} + \text{solar})$$

This is not a data-quality check for its own sake. If the identity holds, then SMARD *defines*
residual load as load minus wind and solar only — which means the project's "wind + solar only"
scope simplification costs **nothing for this target**. Every other generation source (biomass,
coal, hydro, ...) is already outside residual load by construction, not by our choice.

In [ ]:
implied = time_series["grid_load"] - time_series[["wind_off", "wind_on", "solar"]].sum(axis=1)
residual_error = time_series["residual_load"] - implied

abs_err = residual_error.abs()
typical = time_series["residual_load"].abs().median()
exact = (residual_error == 0).mean()
within_1 = (abs_err <= 1).mean()
over_1 = abs_err > 1

print(f"hours checked            : {len(residual_error):,}")
print(f"identity holds exactly   : {exact:6.2%}")
print(f"holds to within 1 MWh    : {within_1:6.2%}")
print(f"max deviation            : {abs_err.max():,.2f} MWh "
      f"({abs_err.max() / typical:.4%} of a typical |residual_load| of {typical:,.0f} MWh)")

if over_1.any():
    print(f"hours deviating > 1 MWh  : {int(over_1.sum())}, confined to "
          f"{residual_error[over_1].index.min():%Y-%m-%d} .. "
          f"{residual_error[over_1].index.max():%Y-%m-%d}")

**The identity holds to within rounding.** It is satisfied exactly for the large majority of
hours and to within 1 MWh for essentially all of them; the handful of larger deviations are
confined to a single short patch of the record and peak at a fraction of a per-mille of a typical
residual-load value — SMARD-side rounding and revision artefacts, not a different definition. At
the scale this notebook thresholds on (tens of thousands of MWh) they are immaterial.

The consequence is the point of the check: SMARD **defines** residual load as grid load minus wind
and solar, so the project's "wind + solar only" scope simplification costs **nothing for this
target**. Biomass, coal, hydro and the rest are outside residual load by construction, not by our
choice.